# LC 1091 — Shortest Path in Binary Matrix
**Day-67 | Theme: Multi-source BFS on Grids | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> BFS on an unweighted grid always yields
the shortest path — mark cells visited <em>when enqueued</em>
(not when dequeued) to avoid re-processing. Eight directions
allow diagonal movement, which is the key twist here.
</div>

## Official Problem Statement

Given an `n x n` binary matrix `grid`, return the length of the
**shortest clear path** from the top-left cell `(0, 0)` to the
bottom-right cell `(n-1, n-1)`.

A clear path is a path where:
- All visited cells have value `0`.
- All adjacent cells in the path are **8-directionally** connected
  (horizontally, vertically, or diagonally adjacent).

The length of a clear path is the **number of visited cells**.
Return `-1` if there is no clear path.

**Constraints:**
- `n == grid.length == grid[i].length`
- `1 <= n <= 100`
- `grid[i][j]` is `0` or `1`

## What This Is Actually Asking

Navigate from the top-left corner to the bottom-right corner of a
binary grid, only stepping on `0` cells, moving in any of 8
directions (including diagonals).
You want the shortest such path, measured by the number of cells
visited — not edges, but cells (so a single-cell path has length 1).
If the start or end cell is blocked (value 1), or no path exists,
return -1.
BFS guarantees the first time you reach the destination is via
the shortest route.

## Walk Through an Example by Hand

```
Input grid (4x4):
  0 0 0 0
  1 1 0 1
  0 0 0 1
  0 0 0 0
```

**Step 1 — Enqueue start:** queue=[(0,0,1)], mark (0,0) visited
  (distance = 1 because we count cells)

**Step 2 — Expand (0,0) dist=1:**
  8 neighbors: (0,1)✓, (1,0)=1✗, (1,1)=1✗  → enqueue (0,1,2)

**Step 3 — Expand (0,1) dist=2:**
  neighbors: (0,2)✓, (1,2)✓  → enqueue both at dist=3

**Step 4 — Expand (0,2) dist=3:**
  neighbors: (0,3)✓, (1,2)visited, (1,3)=1✗ → enqueue (0,3,4)

**Step 5 — Expand (1,2) dist=3:**
  neighbors: (2,1)✓, (2,2)✓, (2,3)=1✗ → enqueue at dist=4

**Step 6 — Expand (0,3) dist=4:**
  no new unblocked neighbors reachable

**BFS continues... eventually (3,3) is reached at dist=6**
Return **6**

Shortest path: (0,0)→(0,1)→(0,2)→(1,2)→(2,2)→(3,3)  (6 cells)

## The Picture

8-directional BFS expanding outward from (0,0):

```
Grid (0=clear, 1=blocked, *=path, d=BFS distance):

  Col:  0    1    2    3
Row 0: [d=1][d=2][d=3][d=4]   ← top row all clear
Row 1: [ 1 ][ 1 ][d=3][ 1 ]   ← wall forces route through (1,2)
Row 2: [   ][d=4][d=4][ 1 ]   ← diagonal from (1,2) fans out
Row 3: [   ][   ][d=5][d=6]   ← (3,3) reached at distance 6

BFS waves (each wave = 1 step):
  Wave 1: {(0,0)}
  Wave 2: {(0,1)}
  Wave 3: {(0,2), (1,2)}
  Wave 4: {(0,3), (2,1), (2,2)}
  Wave 5: {(3,0),(3,1),(3,2),(2,0)}  ← diagonals fan wide
  Wave 6: {(3,3)}  ← DESTINATION REACHED  return 6

8 directions checked at each cell:
  (-1,-1) (-1,0) (-1,+1)
  ( 0,-1) [cell] ( 0,+1)
  (+1,-1) (+1,0) (+1,+1)

Mark visited ON ENQUEUE — prevents same cell entering queue twice!
```

## When To Use This Pattern

- When asked for **shortest path on an unweighted grid**, BFS is
  always the right tool — it explores by distance level.
- When the grid allows **8-directional movement** (diagonals),
  expand your direction list from 4 to all 8 offsets.
- When you must avoid certain cells (value 1), simply skip them
  during neighbor expansion.
- When path length = **cell count** (not edge count), initialize
  distance at 1 for the start cell, not 0.
- When start or end is blocked, short-circuit immediately with -1
  before BFS runs.

## The Approach

Guard against blocked start/end immediately — if `grid[0][0]` or
`grid[n-1][n-1]` is 1, return -1.
Enqueue `(0, 0, 1)` (row, col, distance) and mark `grid[0][0] = 1`
to prevent revisiting.
In BFS, for each cell dequeued, check all 8 neighbors: if in-bounds
and `== 0`, enqueue with `dist + 1` and mark visited.
If the dequeued cell is `(n-1, n-1)`, return its distance
immediately as the shortest path length.

In [ ]:
from typing import List
from collections import deque

In [ ]:
import copy


def test_harness(func):
    """
    Run test cases for LC 1091 — Shortest Path in Binary Matrix.
    Uses copy.deepcopy so original grids are not mutated.
    """
    cases = [
        # (grid, expected, label)
        (
            [[0, 1], [1, 0]],
            2,
            "2x2 diagonal path",
        ),
        (
            [[0, 0, 0], [1, 1, 0], [1, 1, 0]],
            4,
            "3x3 around wall",
        ),
        (
            [[1, 0, 0], [1, 1, 0], [1, 1, 0]],
            -1,
            "start blocked",
        ),
        (
            [[0, 0, 0], [1, 1, 0], [1, 1, 1]],
            -1,
            "end blocked",
        ),
        (
            [[0]],
            1,
            "1x1 grid — start is end",
        ),
        (
            [[1]],
            -1,
            "1x1 blocked",
        ),
        (
            [[0, 0, 0, 0],
             [1, 1, 0, 1],
             [0, 0, 0, 1],
             [0, 0, 0, 0]],
            6,
            "4x4 winding path",
        ),
    ]

    passed = 0
    failed = 0
    for grid, expected, label in cases:
        result = func(copy.deepcopy(grid))
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            failed += 1
        print(
            f"{status} | {label:<35} "
            f"expected={expected} got={result}"
        )

    print(f"\nSummary: {passed} passed, {failed} failed "
          f"out of {passed + failed} tests")

In [ ]:
def shortest_path_binary_matrix(
    grid: List[List[int]]
) -> int:
    """
    Return length of shortest clear path from (0,0) to (n-1,n-1).

    Strategy: BFS with 8-directional movement.
    - Guard: if start or end is 1, return -1 immediately.
    - Enqueue (row, col, distance=1) for cell (0,0).
    - Mark cells visited by setting grid[r][c] = 1 on enqueue.
    - Expand in all 8 directions; skip blocked or visited cells.
    - Return distance when (n-1, n-1) is dequeued.
    - Return -1 if queue empties without reaching destination.

    Args:
        grid: n x n binary grid (0=clear, 1=blocked).

    Returns:
        Length of shortest clear path (cell count), or -1.

    Examples:
        >>> shortest_path_binary_matrix([[0,1],[1,0]])
        2
        >>> shortest_path_binary_matrix([[0,0,0],[1,1,0],[1,1,0]])
        4
        >>> shortest_path_binary_matrix([[1,0]])
        -1
    """
    n = len(grid)
    print(f"[DEBUG] Grid size: {n}x{n}")
    print(f"[DEBUG] Start={grid[0][0]}, End={grid[n-1][n-1]}")

    # TODO: Step 1 — guard blocked start/end
    # TODO: Step 2 — BFS with 8-directional moves
    # TODO: Step 3 — return dist or -1

    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(shortest_path_binary_matrix)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| DFS (wrong — not shortest) | O(n²) | O(n²) | Finds A path, not shortest |
| BFS optimal | O(n²) | O(n²) | Each of n² cells visited once |

**n** = side length of the grid.  
Queue holds at most O(n²) cells — space is O(n²).  
Each cell is enqueued at most once — time is O(n²).  
8 directions is a constant factor, not an asymptotic difference.

## Real World Connection

**Citi / Trading Systems:** Finding the lowest-latency network
route between two data centers in a grid-like topology where some
links are down (blocked); BFS gives the fewest-hop path.

**AWS / Cloud Networking:** VPC routing tables modeled as a grid
where certain subnets are unavailable; shortest path BFS helps
automated failover routing pick the quickest recovery lane.

**Data Engineering:** In a data lineage graph laid out as a matrix,
finding the shortest dependency chain between a raw source table
and a reporting table — critical for understanding blast radius
of schema changes and minimizing reprocessing time.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra